In [1]:
# Data handling
import pandas as pd
import numpy as np

# Visualization (optional for exploration)
import matplotlib.pyplot as plt
import seaborn as sns

# Train-test split
from sklearn.model_selection import train_test_split

# Preprocessing
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Model
from lightgbm import LGBMClassifier

# Evaluation
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Save model
import pickle

print("Libraries imported successfully")

Libraries imported successfully


In [2]:
# Load dataset
df = pd.read_csv("/content/ai-impact-jobs-layoff-risk-dataset.csv")

# Display first 5 rows
df.head()

,Age,Education_Level,Years_of_Experience,Industry,Job_Role,Company_Size,Job_Level,Routine_Task_Percentage,Creativity_Requirement,Human_Interaction_Level,AI_Adoption_Level,Number_of_AI_Tools_Used,AI_Usage_Hours_Per_Week,Tasks_Automated_Percentage,AI_Training_Hours,Layoff_Risk
0,59,Master's,6,Finance,Accountant,Medium,Entry,84,21,94,Medium,5,7,40,26,High
1,44,Master's,14,Manufacturing,Production Supervisor,Small,Entry,30,84,68,Low,2,2,14,9,Low
2,36,Bachelor's,7,Retail,Store Manager,Medium,Senior,12,86,71,Low,0,1,6,3,Low
3,27,Bachelor's,6,Finance,Auditor,Large,Entry,90,18,21,Medium,3,10,67,26,High
4,49,High School,12,Finance,Auditor,Small,Entry,49,52,72,Medium,5,13,26,19,Medium


In [3]:
# Dataset shape (rows, columns)
print("Shape:", df.shape)

# Column names
print("\nColumns:")
print(df.columns)

# Data types
print("\nData Types:")
print(df.dtypes)

# Missing values
print("\nMissing Values:")
print(df.isnull().sum())

Shape: (20000, 16)

Columns:
Index(['Age', 'Education_Level', 'Years_of_Experience', 'Industry', 'Job_Role',
       'Company_Size', 'Job_Level', 'Routine_Task_Percentage',
       'Creativity_Requirement', 'Human_Interaction_Level',
       'AI_Adoption_Level', 'Number_of_AI_Tools_Used',
       'AI_Usage_Hours_Per_Week', 'Tasks_Automated_Percentage',
       'AI_Training_Hours', 'Layoff_Risk'],
      dtype='object')

Data Types:
Age                            int64
Education_Level               object
Years_of_Experience            int64
Industry                      object
Job_Role                      object
Company_Size                  object
Job_Level                     object
Routine_Task_Percentage        int64
Creativity_Requirement         int64
Human_Interaction_Level        int64
AI_Adoption_Level             object
Number_of_AI_Tools_Used        int64
AI_Usage_Hours_Per_Week        int64
Tasks_Automated_Percentage     int64
AI_Training_Hours              int64
Layoff_Risk    

In [4]:
# Separate features and target

X = df.drop("Layoff_Risk", axis=1)

y = df["Layoff_Risk"]

# Check
print("Features shape:", X.shape)
print("Target shape:", y.shape)

print("\nTarget Classes:")
print(y.value_counts())

Features shape: (20000, 15)
Target shape: (20000,)

Target Classes:
Layoff_Risk
High      6797
Low       6602
Medium    6601
Name: count, dtype: int64


In [5]:
# Identify numerical and categorical columns

numerical_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

categorical_features = X.select_dtypes(include=['object']).columns.tolist()

print("Numerical Features:")
print(numerical_features)

print("\nCategorical Features:")
print(categorical_features)

Numerical Features:
['Age', 'Years_of_Experience', 'Routine_Task_Percentage', 'Creativity_Requirement', 'Human_Interaction_Level', 'Number_of_AI_Tools_Used', 'AI_Usage_Hours_Per_Week', 'Tasks_Automated_Percentage', 'AI_Training_Hours']

Categorical Features:
['Education_Level', 'Industry', 'Job_Role', 'Company_Size', 'Job_Level', 'AI_Adoption_Level']


In [6]:
from sklearn.preprocessing import LabelEncoder

# Encode target variable
label_encoder = LabelEncoder()

y_encoded = label_encoder.fit_transform(y)

# Check mapping
print("Classes:", label_encoder.classes_)

print("\nEncoded values:")
print(y_encoded[:10])

Classes: ['High' 'Low' 'Medium']

Encoded values:
[0 1 1 0 2 0 2 2 1 2]


In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

# Check shapes
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (16000, 15)
X_test: (4000, 15)
y_train: (16000,)
y_test: (4000,)


In [8]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

# Preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ],
    remainder="passthrough"
)

print("Preprocessing pipeline created successfully")

Preprocessing pipeline created successfully


In [9]:
from lightgbm import LGBMClassifier
from sklearn.pipeline import Pipeline

# Define LightGBM model
lgbm_model = LGBMClassifier(
    objective="multiclass",
    num_class=3,
    random_state=42
)

# Create complete pipeline
model_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", lgbm_model)
    ]
)

# Train model
model_pipeline.fit(X_train, y_train)

print("Model trained successfully")

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002915 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 638
[LightGBM] [Info] Number of data points in the train set: 16000, number of used features: 54
[LightGBM] [Info] Start training from score -1.079361
[LightGBM] [Info] Start training from score -1.108284
[LightGBM] [Info] Start training from score -1.108473
Model trained successfully


In [10]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Predictions
y_pred = model_pipeline.predict(X_test)

# Accuracy
accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)

# Classification Report
print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=label_encoder.classes_
))

# Confusion Matrix
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))


Accuracy: 0.94675

Classification Report:
              precision    recall  f1-score   support

        High       0.97      0.96      0.96      1360
         Low       0.96      0.95      0.96      1320
      Medium       0.91      0.93      0.92      1320

    accuracy                           0.95      4000
   macro avg       0.95      0.95      0.95      4000
weighted avg       0.95      0.95      0.95      4000


Confusion Matrix:
[[1302    0   58]
 [   0 1259   61]
 [  46   48 1226]]


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [11]:
import pickle

# Save complete model pipeline
with open("ai_job_risk_lgbm_model.pkl", "wb") as file:
    pickle.dump(model_pipeline, file)

# Save label encoder
with open("label_encoder.pkl", "wb") as file:
    pickle.dump(label_encoder, file)

print("Model and encoder saved successfully")

Model and encoder saved successfully
